# 🎓 IELTS Speaking Video Maker + RVC Voice Cloning

한국어 답변 → IELTS 영어 답변 → 내 목소리 MP4 영상 생성

## ⏱️ 총 사용 흐름
1. **Cell 1**: RVC 환경 설치 (최초 1회, ~10분)
2. **Cell 2**: 내 음성 학습 (최초 1회, ~20분)
3. **Cell 3**: 메인 앱 - 'Generate Video' 버튼만 클릭!

> **주의**: 반드시 **Runtime → Change runtime type → T4 GPU** 선택


In [ ]:
# @title Cell 1: Setup Environment (RVC + IELTS Base)
import os, subprocess, urllib.request, zipfile, torch

def sh(cmd, label=None):
    tag = label or cmd[:60]
    r = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(f'\n❌ Error [{tag}]')
        print(r.stdout[-1200:])
        raise RuntimeError(tag)
    return r.stdout

try:
    print('🔧 === Cell 1: Environment Setup ===\n')

    # 1. GPU Check
    print('1/7 GPU Check...')
    print(f'   CUDA: {torch.cuda.is_available()}', end='')
    if torch.cuda.is_available():
        print(f' ({torch.cuda.get_device_name(0)})')
    else:
        print('\n   ERROR: No GPU! Runtime → Change runtime type → T4 GPU')
        raise RuntimeError('No GPU')

    # 2. System Packages
    print('2/7 System Packages...')
    sh('apt-get update -q')
    sh('apt-get install -y ffmpeg build-essential -q', 'apt install')

    # 3. pip Upgrade
    print('3/7 pip Upgrade...')
    sh('pip install --upgrade pip setuptools wheel -q')

    # 4. IELTS Packages
    print('4/7 IELTS Packages...')
    sh('pip install "moviepy<2.0" Pillow ipywidgets -q', 'moviepy<2.0')
    sh('pip install "google-generativeai>=0.7" -q', 'google-generativeai')

    # 5. RVC Clone
    print('5/7 RVC Repository...')
    if not os.path.exists('/content/RVC'):
        sh('git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/RVC -q', 'git clone')
    else:
        print('   (Existing)')

    # 6. RVC Packages
    print('6/7 RVC Packages... (~3min)')
    os.chdir('/content/RVC')
    sh('pip install -q -r requirements.txt 2>/dev/null || true', 'requirements')
    sh('pip install -q faiss-gpu fairseq gradio edge-tts torchcrepe', 'RVC extras')

    # 7. Download Models
    print('7/7 Download Models... (~3min)')
    for dname in ['assets/hubert', 'assets/rmvpe', 'assets/pretrained_v2', 'logs']:
        os.makedirs(dname, exist_ok=True)

    models = [
        ('assets/hubert/hubert_base.pt',
         'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt'),
        ('assets/rmvpe/rmvpe.pt',
         'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt'),
        ('assets/pretrained_v2/f0G40k.pth',
         'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth'),
        ('assets/pretrained_v2/f0D40k.pth',
         'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth'),
    ]
    for local_path, url in models:
        if not os.path.exists(local_path):
            print(f'   Download {os.path.basename(local_path)}...')
            urllib.request.urlretrieve(url, local_path)
        else:
            print(f'   ✓ {os.path.basename(local_path)}')

    print('\n✅ Cell 1 Complete!')
    print('   → Go to Cell 2\n')

except RuntimeError as e:
    print(f'\n❌ Stopped: {e}')
    raise


In [ ]:
# @title Cell 2: Train Your Voice Model
import os, sys, subprocess
from google.colab import files

os.chdir('/content/RVC')

print('🎤 === Cell 2: Voice Training ===\n')

VOICE_MODEL_NAME = "my_voice"
DATASET_DIR = f"/content/RVC/dataset/{VOICE_MODEL_NAME}"
os.makedirs(DATASET_DIR, exist_ok=True)

print('📂 Step 1: Upload Voice Files')
print('   Recommended: 10~30 min English audio (MP3/WAV)\n')

try:
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = os.path.join(DATASET_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(data)
        size_mb = len(data) / 1024 / 1024
        print(f'   ✓ {fname} ({size_mb:.1f}MB)')
    print()
except Exception as e:
    print(f'   ❌ Error: {e}')
    raise

SAMPLE_RATE = 40000
N_CPU = 4

print('✂️ Step 2: Audio Preprocessing... (~3min)')
cmd = f'python trainset_preprocess_pipeline_print.py "{DATASET_DIR}" {SAMPLE_RATE} {N_CPU} "logs/{VOICE_MODEL_NAME}" True'
r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if r.returncode != 0:
    print(f'   ❌ Error:\n{r.stdout[-600:]}')
    raise RuntimeError('Preprocessing')
print('   ✓ Complete\n')

print('🔍 Step 3: Feature Extraction... (~5min)')
cmd = f'python extract_f0_print.py "logs/{VOICE_MODEL_NAME}" {N_CPU} "rmvpe"'
subprocess.run(cmd, shell=True)
cmd = f'python extract_feature_print.py cuda:0 1 0 0 "logs/{VOICE_MODEL_NAME}" v2'
subprocess.run(cmd, shell=True)
print('   ✓ Complete\n')

TOTAL_EPOCH = 200
BATCH_SIZE = 7
SAVE_EVERY = 50

print(f'🏋️ Step 4: Model Training... (~15-30min, {TOTAL_EPOCH} epochs)')
print('   DO NOT interrupt.\n')

LOG_DIR = f"logs/{VOICE_MODEL_NAME}"
script = f'''
import os
log_dir = "{LOG_DIR}"
gt_wav_dir = os.path.join(log_dir, "0_gt_wavs")
feat_dir = os.path.join(log_dir, "3_feature256")
f0_dir = os.path.join(log_dir, "2a_f0")
nsf_dir = os.path.join(log_dir, "2b-f0nsf")
out_path = os.path.join(log_dir, "filelist.txt")
lines = []
for fname in os.listdir(gt_wav_dir):
    if fname.endswith(".wav"):
        name = fname[:-4]
        wav = os.path.join(gt_wav_dir, fname)
        feat = os.path.join(feat_dir, name + ".npy")
        f0 = os.path.join(f0_dir, name + ".wav.npy")
        nsf = os.path.join(nsf_dir, name + ".wav.npy")
        if os.path.exists(feat) and os.path.exists(f0) and os.path.exists(nsf):
            lines.append(wav + "|" + feat + "|" + f0 + "|" + nsf)
with open(out_path, "w") as fp:
    fp.write("\n".join(lines))
print(f"Filelist: {len(lines)} entries")
'''
with open('/tmp/make_filelist.py', 'w') as fp:
    fp.write(script)
subprocess.run('python3 /tmp/make_filelist.py', shell=True)

cmd = f'python train_nsf_sim_cache_sid_load_pretrain.py -e "{VOICE_MODEL_NAME}" -sr {SAMPLE_RATE} -f0 1 -bs {BATCH_SIZE} -g 0 -te {TOTAL_EPOCH} -se {SAVE_EVERY} -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth -l 1 -c 0 -sw 1 -v v2'
r = subprocess.run(cmd, shell=True)
if r.returncode != 0:
    print('❌ Training failed')
    raise RuntimeError('Training')

print('✓ Complete\n')

print('📊 Step 5: FAISS Index...')
script = f'''
import numpy as np, os, faiss
model_name = "{VOICE_MODEL_NAME}"
log_dir = "logs/{VOICE_MODEL_NAME}"
feature_dir = os.path.join(log_dir, "3_feature256")
if os.path.exists(feature_dir):
    npys = []
    for fname in sorted(os.listdir(feature_dir)):
        if fname.endswith(".npy"):
            npys.append(np.load(os.path.join(feature_dir, fname)))
    if npys:
        big_npy = np.concatenate(npys, axis=0)
        n_ivf = min(int(16 * np.sqrt(big_npy.shape[0])), big_npy.shape[0] // 39)
        index = faiss.index_factory(256, "IVF" + str(n_ivf) + ",Flat")
        index.train(big_npy)
        index.add(big_npy)
        index_path = os.path.join(log_dir, f"trained_IVF{n_ivf}_Flat_nprobe_1_{model_name}_v2.index")
        faiss.write_index(index, index_path)
        print(f"Index: {index_path}")
'''
with open('/tmp/make_faiss.py', 'w') as fp:
    fp.write(script)
subprocess.run('python3 /tmp/make_faiss.py', shell=True)

print('✓ Complete\n')
print('🎉 Cell 2 Done!')
print('   → Go to Cell 3\n')


In [ ]:
# @title Cell 3: IELTS Video Maker (RVC Voice TTS)
import os, sys, json, tempfile, warnings, asyncio, glob
import torch, numpy as np
import ipywidgets as w
from IPython.display import display, clear_output, HTML
from PIL import Image, ImageDraw, ImageFont
import edge_tts

os.chdir('/content/RVC')
sys.path.insert(0, '/content/RVC')

VOICE_MODEL_NAME = "my_voice"
RVC_SAMPLE_RATE = 40000
RVC_F0_METHOD = "rmvpe"
warnings.filterwarnings('ignore')

QUESTIONS = {
    'Part 1 - Daily Life': [
        'Tell me about your hometown. What do you like most about it?',
        'Do you enjoy cooking? Why or why not?',
        'How do you usually spend your weekends?',
        'What kind of music do you like? Why?',
        'Do you prefer to study in the morning or at night?',
    ],
    'Part 1 - Hobbies': [
        'What hobbies do you have?',
        'Do you enjoy reading books? What kind?',
        'How often do you exercise?',
    ],
    'Part 2 - Experience': [
        'Describe a memorable trip you have taken.',
        'Describe a time when you helped someone.',
        'Describe an achievement you are proud of.',
    ],
    'Part 3 - Society': [
        'Do you think technology has improved education? In what ways?',
        'How important is it for young people to learn a second language?',
        'What are the advantages and disadvantages of working from home?',
    ],
}
SCORE_DESC = {
    5.0: 'Basic - simple sentences',
    5.5: 'Elementary - more natural flow',
    6.0: 'Intermediate - complex structures',
    6.5: 'Upper-intermediate - varied vocabulary',
    7.0: 'Advanced - idioms included',
    7.5: 'Proficient - near-native',
    8.0: 'Expert - native level',
}
STYLE_MAP = {
    'Dark Minimal': ((15,15,25),(196,149,106),(235,235,235),(150,150,160)),
    'Light Clean': ((248,246,241),(70,90,160),(30,30,40),(120,120,130)),
    'Deep Blue': ((10,30,60),(100,180,255),(230,240,255),(140,170,210)),
}

state = dict(models_loaded=False, rvc_model_path=None, rvc_index_path=None)

CSS = '<style>.il-title{font-size:20px;font-weight:600;color:#1a1a1a;margin:0 0 4px}.il-sub{font-size:13px;color:#888;margin-bottom:18px}.il-label{font-size:11px;font-weight:700;color:#aaa;margin:14px 0 4px}.il-result{background:#eef4fb;border:1px solid #b8d4ee;border-radius:8px;padding:14px;font-size:13px;line-height:1.8;white-space:pre-wrap}.il-tip{background:#f0faf4;border:1px solid #a8dbb8;border-radius:8px;padding:12px;font-size:13px;color:#2d6a4a}.il-ok{color:#2d7a4a;font-weight:600}.il-err{color:#c0392b;font-weight:600}.il-warn{color:#c07a00;font-weight:600}hr.il{border:none;border-top:1px solid #eee;margin:14px 0}</style>'
display(HTML(CSS))
display(HTML('<div class="il-title">IELTS Speaking Video Maker</div><div class="il-sub">Korean → English → My Voice MP4</div>'))

display(HTML('<div class="il-label">Gemini API Key</div>'))
w_key = w.Password(placeholder='aistudio.google.com', layout=w.Layout(width='440px'))
display(w_key)

display(HTML('<hr class="il"><div class="il-label">IELTS Question</div>'))
w_part = w.Dropdown(options=list(QUESTIONS.keys()), layout=w.Layout(width='240px'))
w_q = w.Dropdown(options=QUESTIONS[w_part.value], layout=w.Layout(width='460px'))
w_part.observe(lambda c: setattr(w_q, 'options', QUESTIONS[c['new']]), names='value')
display(w_part, w_q)

display(HTML('<div class="il-label" style="margin-top:10px">Korean Answer</div>'))
w_korean = w.Textarea(placeholder='Korean text...', layout=w.Layout(width='460px', height='110px'))
display(w_korean)

display(HTML('<hr class="il"><div class="il-label">Target IELTS Band</div>'))
w_score = w.SelectionSlider(options=[5.0,5.5,6.0,6.5,7.0,7.5,8.0], value=6.5, layout=w.Layout(width='320px'))
out_sc = w.Output()
w_score.observe(lambda c: display(HTML(f'<span style="font-size:12px;color:#888">{SCORE_DESC[c["new"]]}</span>')) if out_sc.outputs else None, names='value')
display(w_score, out_sc)

display(HTML('<hr class="il"><div class="il-label">Voice Settings</div>'))
w_base_voice = w.Dropdown(options=['en-US-GuyNeural', 'en-US-JennyNeural', 'en-GB-RyanNeural', 'en-GB-SoniaNeural'], value='en-US-GuyNeural', layout=w.Layout(width='240px'))
display(w_base_voice)
w_f0 = w.IntSlider(value=0, min=-12, max=12, step=1, description='Pitch:', layout=w.Layout(width='320px'))
w_idx = w.FloatSlider(value=0.75, min=0, max=1, step=0.05, description='Voice Sim:', layout=w.Layout(width='320px'))
display(w_f0, w_idx)

display(HTML('<div class="il-label" style="margin-top:10px">Video Style</div>'))
w_style = w.ToggleButtons(options=['Dark Minimal','Light Clean','Deep Blue'], value='Dark Minimal')
display(w_style)

display(HTML('<hr class="il">'))
btn = w.Button(description='Generate Video', button_style='success', layout=w.Layout(width='200px', height='44px'))
out_main = w.Output()
display(btn, out_main)

def find_rvc_models():
    log_dir = f"logs/{VOICE_MODEL_NAME}"
    pth = sorted(glob.glob(f"{log_dir}/*.pth"), key=os.path.getmtime, reverse=True)
    idx = sorted(glob.glob(f"{log_dir}/*.index"), key=os.path.getmtime, reverse=True)
    return (pth[0] if pth else None, idx[0] if idx else None)

def call_gemini(question, korean, score, api_key):
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-2.0-flash')
    prompt = f'You are IELTS coach. Make English (Band {score}) from Korean.\nBand {score}: {SCORE_DESC[score]}\nQ: {question}\nKorean: {korean}\nJSON: {{"english":"...","korean":"...","band_tips":"..."}}'
    raw = model.generate_content(prompt).text.strip()
    if '```' in raw:
        raw = raw.split('```')[1] if len(raw.split('```')) > 1 else raw
        if raw.startswith('json'): raw = raw[4:]
    s, e = raw.find('{'), raw.rfind('}')
    if s == -1 or e == -1:
        raise ValueError('No JSON')
    return json.loads(raw[s:e+1])

async def gen_tts(text, voice, path):
    await edge_tts.Communicate(text, voice).save(path)

def rvc_convert(input_p, output_p, model_p, idx_p, f0, idx_r):
    cmd = f'python infer_cli.py --model_path "{model_p}" --input_path "{input_p}" --output_path "{output_p}" --f0method rmvpe --f0up_key {f0} --index_rate {idx_r} --protect 0.33'
    if idx_p:
        cmd = cmd.replace('--output_path', f'--index_path "{idx_p}" --output_path')
    return os.system(cmd + ' >/dev/null 2>&1') == 0

def make_frame(style, label, text, score):
    W, H = 1280, 720
    img = Image.new('RGB', (W, H))
    draw = ImageDraw.Draw(img)
    bg, ac, fg, mu = STYLE_MAP.get(style, STYLE_MAP['Dark Minimal'])
    draw.rectangle([0,0,W,H], fill=bg)
    draw.rectangle([60,80,80,H-80], fill=ac)
    try:
        fB = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 18)
        fM = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 28)
        fS = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 14)
    except:
        fB = fM = fS = ImageFont.load_default()
    draw.text((110,90), label.upper(), fill=ac, font=fB)
    words = text.split()
    lines, cur = [], []
    for wd in words:
        cur.append(wd)
        if len(' '.join(cur)) > 52:
            lines.append(' '.join(cur[:-1]))
            cur = [wd]
    lines.append(' '.join(cur))
    y = 140
    for line in lines[:6]:
        draw.text((110,y), line, fill=fg, font=fM)
        y += 44
    draw.text((W-230, H-50), f'IELTS Band {score}', fill=mu, font=fS)
    return np.array(img)

def on_click(b):
    with out_main:
        clear_output()
        api_key = w_key.value.strip()
        if not api_key:
            display(HTML('<span class="il-err">No API key</span>'))
            return
        if not w_korean.value.strip():
            display(HTML('<span class="il-err">No Korean text</span>'))
            return

        tmpdir = tempfile.mkdtemp()
        if not state['models_loaded']:
            display(HTML('<span class="il-warn">Finding RVC model...</span>'))
            mp, ip = find_rvc_models()
            if not mp:
                display(HTML('<span class="il-err">No model. Run Cell 2.</span>'))
                return
            state['rvc_model_path'], state['rvc_index_path'] = mp, ip
            state['models_loaded'] = True
            display(HTML(f'<span class="il-ok">✓ {os.path.basename(mp)}</span>'))

        display(HTML('<span class="il-warn">Generating English...</span>'))
        try:
            r = call_gemini(w_q.value, w_korean.value.strip(), w_score.value, api_key)
            eng = r['english']
            display(HTML('<div class="il-result">' + eng + '</div>'))
            display(HTML('<div class="il-tip">' + r['korean'] + '<br><br>' + r['band_tips'] + '</div>'))
        except Exception as e:
            display(HTML(f'<span class="il-err">Error: {e}</span>'))
            return

        display(HTML('<span class="il-warn">Generating base voice...</span>'))
        bq, ba = os.path.join(tmpdir, 'bq.wav'), os.path.join(tmpdir, 'ba.wav')
        try:
            asyncio.run(gen_tts(w_q.value, w_base_voice.value, bq))
            asyncio.run(gen_tts(eng, w_base_voice.value, ba))
            display(HTML('<span class="il-ok">✓ Base voice done</span>'))
        except Exception as e:
            display(HTML(f'<span class="il-err">Error: {e}</span>'))
            return

        display(HTML('<span class="il-warn">RVC conversion (1-2min)...</span>'))
        rq, ra = os.path.join(tmpdir, 'rq.wav'), os.path.join(tmpdir, 'ra.wav')
        try:
            if not rvc_convert(bq, rq, state['rvc_model_path'], state['rvc_index_path'], w_f0.value, w_idx.value):
                raise RuntimeError('Q failed')
            if not rvc_convert(ba, ra, state['rvc_model_path'], state['rvc_index_path'], w_f0.value, w_idx.value):
                raise RuntimeError('A failed')
            display(HTML('<span class="il-ok">✓ Voice conversion done</span>'))
        except Exception as e:
            display(HTML(f'<span class="il-err">Error: {e}</span>'))
            return

        display(HTML('<span class="il-warn">Compositing video...</span>'))
        from moviepy.editor import AudioFileClip, ImageClip, concatenate_videoclips
        out_mp4 = '/content/ielts_speaking.mp4'
        try:
            qf = make_frame(w_style.value, 'Question', w_q.value, w_score.value)
            af = make_frame(w_style.value, 'Answer', eng, w_score.value)
            qa = AudioFileClip(rq)
            aa = AudioFileClip(ra)
            qc = ImageClip(qf).set_duration(qa.duration + 1.5).set_audio(qa)
            ac = ImageClip(af).set_duration(aa.duration + 2.0).set_audio(aa)
            final = concatenate_videoclips([qc, ac], method='compose')
            final.write_videofile(out_mp4, fps=24, codec='libx264', audio_codec='aac', logger=None)
            qa.close(); aa.close(); final.close()
            display(HTML('<span class="il-ok">✓ Video done!</span>'))
        except Exception as e:
            display(HTML(f'<span class="il-err">Error: {e}</span>'))
            return

        from google.colab import files
        files.download(out_mp4)
        display(HTML('<span class="il-ok">✅ Download started!</span>'))

btn.on_click(on_click)
